## Python-Only SFT Data Pipeline
Build the final SFT dataset for a 100M-param Python coding model.


## 0. Setup

In [1]:
!pip install datasets --quiet

In [2]:
import ast
import hashlib
import json
import re
from collections import Counter

from datasets import load_dataset

CODE_BLOCK_RE = re.compile(r"```(?:python)?\s*(.*?)```", re.S)
print("Imports OK")

Imports OK


### 1. Utility functions

- `extract_code`: pulls code out of a markdown fence if present
- `is_valid_python`: AST-based validation (not regex) — rejects non-Python, truncated, or corrupted samples
- `normalize_for_dedup`: strips comments/docstrings/whitespace for near-dup hashing

In [3]:
def extract_code(text: str) -> str:
    if not text:
        return ""
    m = CODE_BLOCK_RE.search(text)
    return m.group(1).strip() if m else text.strip()


def is_valid_python(code: str) -> bool:
    if not code or len(code) < 10:
        return False
    try:
        ast.parse(code)
        return True
    except Exception:
        return False


def normalize_for_dedup(code: str) -> str:
    code = re.sub(r"#.*", "", code)
    code = re.sub(r'"""[\s\S]*?"""', "", code)
    code = re.sub(r"'''[\s\S]*?'''", "", code)
    code = re.sub(r"\s+", " ", code).strip().lower()
    return code


def make_record(problem, solution, tests=None, source="", layer=""):
    text = f"### Problem\n{problem}\n\n### Solution\n```python\n{solution}\n```"
    return {
        "text": text,
        "problem": problem,
        "solution": solution,
        "tests": tests,
        "source": source,
        "layer": layer,  # "core" or "diversity"
    }

print("Utility functions defined")

Utility functions defined


### 2. Configtarget sizes per source

Adjust these if your first run shows you're data-starved or over-budget on tokens.

In [4]:
CONFIG = {
    "opencodeinstruct_n": 100_000,
    "sc2_n": 50_000,
    "tested143k_n": 70_000,
    "opencoder_sft_n": 50_000,
    "magicoder_evol_n": 45_000,
    "skip_decontamination": False,   # set True to skip the HumanEval/MBPP check (not recommended)
    "out_path": "sft_mix_final.jsonl",
}
CONFIG

{'opencodeinstruct_n': 100000,
 'sc2_n': 50000,
 'tested143k_n': 70000,
 'opencoder_sft_n': 50000,
 'magicoder_evol_n': 45000,
 'skip_decontamination': False,
 'out_path': 'sft_mix_final.jsonl'}

### 3. Core layer — execution-verified sources

### 3a. OpenCodeInstruct
Filtered to `tests_execution_status == pass` only — this is the whole point of using it over an unfiltered dataset.

In [5]:
def load_opencodeinstruct(max_n):
    ds = load_dataset("nvidia/OpenCodeInstruct", split="train")
    out = []
    for row in ds:
        status = str(row.get("tests_execution_status", "")).lower()
        if "pass" not in status:
            continue
        problem = row.get("input", "")
        solution = extract_code(row.get("output", ""))
        if not (problem and is_valid_python(solution)):
            continue
        out.append(make_record(problem, solution, row.get("unit_tests"),
                                source="opencodeinstruct", layer="core"))
        if len(out) >= max_n:
            break
    return out

opencodeinstruct_records = load_opencodeinstruct(CONFIG["opencodeinstruct_n"])
print(f"OpenCodeInstruct: {len(opencodeinstruct_records)} valid records")

README.md:   0%|          | 0.00/4.38k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

data/train-00000-of-00050.parquet:   0%|          | 0.00/133M [00:00<?, ?B/s]

data/train-00001-of-00050.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

...

data/train-00049-of-00050.parquet:   0%|          | 0.00/139M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5000000 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/25 [00:00<?, ?it/s]

<unknown>:10: SyntaxWarning: invalid escape sequence '\.'
<unknown>:10: SyntaxWarning: invalid escape sequence '\d'
<unknown>:13: SyntaxWarning: invalid escape sequence '\W'
<unknown>:13: SyntaxWarning: invalid escape sequence '\W'
<unknown>:3: SyntaxWarning: invalid escape sequence '\|'
<unknown>:9: SyntaxWarning: invalid escape sequence '\,'
<unknown>:14: SyntaxWarning: invalid escape sequence '\w'
<unknown>:28: SyntaxWarning: invalid escape sequence '\.'
<unknown>:13: SyntaxWarning: invalid escape sequence '\W'
<unknown>:3: SyntaxWarning: invalid escape sequence '\z'
<unknown>:37: SyntaxWarning: invalid escape sequence '\d'
<unknown>:38: SyntaxWarning: invalid escape sequence '\.'
<unknown>:38: SyntaxWarning: invalid escape sequence '\.'
<unknown>:7: SyntaxWarning: invalid escape sequence '\,'
<unknown>:9: SyntaxWarning: invalid escape sequence '\.'
<unknown>:21: SyntaxWarning: invalid escape sequence '\.'
<unknown>:16: SyntaxWarning: invalid escape sequence '\*'
<unknown>:18: Synta

OpenCodeInstruct: 100000 valid records


### 3b. self-oss-instruct-sc2-exec-filter-50k
Already execution-filtered upstream, Python-seeded.

In [6]:
def load_sc2_exec_filter(max_n):
    ds = load_dataset("bigcode/self-oss-instruct-sc2-exec-filter-50k", split="train")
    out = []
    for row in ds:
        problem = row.get("instruction", "")
        solution = extract_code(row.get("response", ""))
        if not (problem and is_valid_python(solution)):
            continue
        out.append(make_record(problem, solution, source="sc2_exec_filter", layer="core"))
        if len(out) >= max_n:
            break
    return out

sc2_records = load_sc2_exec_filter(CONFIG["sc2_n"])
print(f"sc2-exec-filter: {len(sc2_records)} valid records")

README.md:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/90.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50661 [00:00<?, ? examples/s]

<unknown>:4: SyntaxWarning: invalid escape sequence '\.'
<unknown>:5: SyntaxWarning: invalid escape sequence '\W'
<unknown>:6: SyntaxWarning: invalid escape sequence '\d'
<unknown>:2: SyntaxWarning: invalid escape sequence '\.'
<unknown>:6: SyntaxWarning: invalid escape sequence '\,'
<unknown>:7: SyntaxWarning: invalid escape sequence '\p'
<unknown>:4: SyntaxWarning: invalid escape sequence '\['
<unknown>:4: SyntaxWarning: invalid escape sequence '\]'
<unknown>:4: SyntaxWarning: invalid escape sequence '\]'
<unknown>:4: SyntaxWarning: invalid escape sequence '\.'
<unknown>:5: SyntaxWarning: invalid escape sequence '\d'
<unknown>:4: SyntaxWarning: invalid escape sequence '\d'
<unknown>:4: SyntaxWarning: invalid escape sequence '\d'
<unknown>:4: SyntaxWarning: invalid escape sequence '\d'
<unknown>:14: SyntaxWarning: invalid escape sequence '\D'
<unknown>:11: SyntaxWarning: invalid escape sequence '\$'
<unknown>:4: SyntaxWarning: invalid escape sequence '\d'
<unknown>:4: SyntaxWarning: i

sc2-exec-filter: 50000 valid records


<unknown>:10: SyntaxWarning: invalid escape sequence '\d'


### 3c. Tested-143k-Python-Alpaca

In [7]:
def load_tested_143k(max_n):
    ds = load_dataset("Vezora/Tested-143k-Python-Alpaca", split="train")
    out = []
    for row in ds:
        problem = row.get("instruction", "")
        extra_input = row.get("input", "")
        if extra_input:
            problem = f"{problem}\n\nInput details: {extra_input}"
        solution = extract_code(row.get("output", ""))
        if not (problem and is_valid_python(solution)):
            continue
        out.append(make_record(problem, solution, source="tested_143k", layer="core"))
        if len(out) >= max_n:
            break
    return out

tested143k_records = load_tested_143k(CONFIG["tested143k_n"])
print(f"Tested-143k-Python-Alpaca: {len(tested143k_records)} valid records")

README.md:   0%|          | 0.00/4.16k [00:00<?, ?B/s]

143k-Tested-Python-Alpaca-Vezora.json:   0%|          | 0.00/295M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/143327 [00:00<?, ? examples/s]

<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:6: SyntaxWarning: invalid escape sequence '\w'
<unknown>:16: SyntaxWarning: invalid escape sequence '\.'
<unknown>:14: SyntaxWarning: invalid escape sequence '\Z'
<unknown>:6: SyntaxWarning: invalid escape sequence '\w'
<unknown>:9: SyntaxWarning: invalid escape sequence '\w'
<unknown>:4: SyntaxWarning: invalid escape sequence '\('
<unknown>:4: SyntaxWarning: invalid escape sequence '\_'
<unknown>:5: SyntaxWarning: invalid escape sequence '\s'
<unknown>:4: SyntaxWarning: invalid escape sequence '\d'
<unknown>:4: SyntaxWarning: invalid escape sequence '\d'
<unknown>:40: SyntaxWarning: invalid escape sequence '\{'
<unknown>:12: SyntaxWarning: invalid escape sequence '\d'
<unknown>:5: SyntaxWarning: invalid escape sequence '\['
<unknown>:5: SyntaxWarning: invalid escape sequence '\s'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\W'
<unknown>:4: SyntaxWarning:

Tested-143k-Python-Alpaca: 70000 valid records


<unknown>:4: SyntaxWarning: invalid escape sequence '\d'
<unknown>:18: SyntaxWarning: invalid escape sequence '\.'
<unknown>:13: SyntaxWarning: invalid escape sequence '\w'


In [8]:
core_records = opencodeinstruct_records + sc2_records + tested143k_records
print(f"Total core layer: {len(core_records)} records")

Total core layer: 220000 records


### 4. Diversity layer

### 4a. OpenCoder-SFT — `educational_instruct` + `package_instruct` ONLY
Deliberately **excludes** the `evol_instruct` subset, since it duplicates Magicoder-Evol-Instruct-110K (same lineage). `package_instruct` is specifically the library/API-usage coverage the core layer is missing.

In [9]:
def load_opencoder_sft_diversity(max_n):
    out = []
    for config_name in ["educational_instruct", "package_instruct"]:
        try:
            ds = load_dataset("OpenCoder-LLM/opc-sft-stage2", config_name, split="train")
        except Exception as e:
            print(f"  SKIPPED opencoder-sft/{config_name}: {e}")
            continue
        cap = max_n // 2
        n_this_config = 0
        for row in ds:
            problem = row.get("instruction", "") or row.get("question", "")
            solution = extract_code(row.get("output", "") or row.get("response", ""))
            if not (problem and is_valid_python(solution)):
                continue
            out.append(make_record(problem, solution,
                                    source=f"opencoder_sft_{config_name}", layer="diversity"))
            n_this_config += 1
            if n_this_config >= cap:
                break
    return out

opencoder_sft_records = load_opencoder_sft_diversity(CONFIG["opencoder_sft_n"])
print(f"OpenCoder-SFT (educational + package): {len(opencoder_sft_records)} valid records")

README.md:   0%|          | 0.00/4.69k [00:00<?, ?B/s]

educational_instruct/train-00000-of-0000(…):   0%|          | 0.00/53.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/118278 [00:00<?, ? examples/s]

<unknown>:14: SyntaxWarning: invalid escape sequence '\s'
<unknown>:14: SyntaxWarning: invalid escape sequence '\s'
<unknown>:6: SyntaxWarning: invalid escape sequence '\d'
<unknown>:11: SyntaxWarning: invalid escape sequence '\|'
<unknown>:5: SyntaxWarning: invalid escape sequence '\w'
<unknown>:7: SyntaxWarning: invalid escape sequence '\w'
<unknown>:11: SyntaxWarning: invalid escape sequence '\w'
<unknown>:12: SyntaxWarning: invalid escape sequence '\-'
<unknown>:4: SyntaxWarning: invalid escape sequence '\.'
<unknown>:6: SyntaxWarning: invalid escape sequence '\W'
<unknown>:4: SyntaxWarning: invalid escape sequence '\W'
<unknown>:4: SyntaxWarning: invalid escape sequence '\.'
<unknown>:12: SyntaxWarning: invalid escape sequence '\s'
<unknown>:4: SyntaxWarning: invalid escape sequence '\.'
<unknown>:4: SyntaxWarning: invalid escape sequence '\d'
<unknown>:7: SyntaxWarning: invalid escape sequence '\d'


package_instruct/train-00000-of-00002.pa(…):   0%|          | 0.00/132M [00:00<?, ?B/s]

package_instruct/train-00001-of-00002.pa(…):   0%|          | 0.00/154M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/170943 [00:00<?, ? examples/s]

<unknown>:13: SyntaxWarning: invalid escape sequence '\('
<unknown>:11: SyntaxWarning: invalid escape sequence '\['
<unknown>:5: SyntaxWarning: invalid escape sequence '\|'
<unknown>:6: SyntaxWarning: invalid escape sequence '\|'
<unknown>:16: SyntaxWarning: invalid escape sequence '\c'
<unknown>:8: SyntaxWarning: invalid escape sequence '\s'
<unknown>:5: SyntaxWarning: invalid escape sequence '\d'
<unknown>:6: SyntaxWarning: invalid escape sequence '\w'
<unknown>:46: SyntaxWarning: invalid escape sequence '\M'
<unknown>:14: SyntaxWarning: invalid escape sequence '\.'
<unknown>:9: SyntaxWarning: invalid escape sequence '\d'
<unknown>:14: SyntaxWarning: invalid escape sequence '\d'
<unknown>:14: SyntaxWarning: invalid escape sequence '\+'
<unknown>:12: SyntaxWarning: invalid escape sequence '\S'
<unknown>:7: SyntaxWarning: invalid escape sequence '\s'
<unknown>:5: SyntaxWarning: invalid escape sequence '\-'
<unknown>:13: SyntaxWarning: invalid escape sequence '\s'
<unknown>:16: SyntaxWa

OpenCoder-SFT (educational + package): 50000 valid records


<unknown>:10: SyntaxWarning: invalid escape sequence '\W'
<unknown>:12: SyntaxWarning: invalid escape sequence '\d'
<unknown>:24: SyntaxWarning: invalid escape sequence '\/'
<unknown>:9: SyntaxWarning: invalid escape sequence '\('


### 4b. Magicoder-Evol-Instruct-110K
Hard algorithmic/reasoning tail. Not execution-verified — kept as a lower-weight diversity source, not core.

In [10]:
def load_magicoder_evol_instruct(max_n):
    ds = load_dataset("ise-uiuc/Magicoder-Evol-Instruct-110K", split="train")
    out = []
    for row in ds:
        problem = row.get("instruction", "")
        solution = extract_code(row.get("response", "") or row.get("output", ""))
        if not (problem and is_valid_python(solution)):
            continue
        out.append(make_record(problem, solution,
                                source="magicoder_evol_instruct", layer="diversity"))
        if len(out) >= max_n:
            break
    return out

magicoder_records = load_magicoder_evol_instruct(CONFIG["magicoder_evol_n"])
print(f"Magicoder-Evol-Instruct: {len(magicoder_records)} valid records")

README.md:   0%|          | 0.00/394 [00:00<?, ?B/s]

data-evol_instruct-decontaminated.jsonl:   0%|          | 0.00/255M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/111183 [00:00<?, ? examples/s]

<unknown>:2: SyntaxWarning: invalid escape sequence '\,'
<unknown>:6: SyntaxWarning: invalid escape sequence '\w'
<unknown>:6: SyntaxWarning: invalid escape sequence '\w'
<unknown>:6: SyntaxWarning: invalid escape sequence '\d'
<unknown>:19: SyntaxWarning: invalid escape sequence '\w'
<unknown>:26: SyntaxWarning: invalid escape sequence '\('
<unknown>:5: SyntaxWarning: invalid escape sequence '\.'
<unknown>:18: SyntaxWarning: invalid escape sequence '\d'
<unknown>:32: SyntaxWarning: invalid escape sequence '\.'
<unknown>:74: SyntaxWarning: invalid escape sequence '\d'
<unknown>:15: SyntaxWarning: invalid escape sequence '\w'
<unknown>:18: SyntaxWarning: invalid escape sequence '\S'
<unknown>:26: SyntaxWarning: invalid escape sequence '\d'
<unknown>:4: SyntaxWarning: invalid escape sequence '\.'
<unknown>:24: SyntaxWarning: invalid escape sequence '\('
<unknown>:29: SyntaxWarning: invalid escape sequence '\.'
<unknown>:8: SyntaxWarning: invalid escape sequence '\s'
<unknown>:4: SyntaxWa

Magicoder-Evol-Instruct: 45000 valid records


In [11]:
diversity_records = opencoder_sft_records + magicoder_records
print(f"Total diversity layer: {len(diversity_records)} records")

Total diversity layer: 95000 records


## 5. Combine, deduplicate

Global exact/near-dup removal across ALL five sources using a normalized-code hash (strips comments/docstrings/whitespace before hashing, so trivial formatting differences still count as duplicates).

In [12]:
all_records = core_records + diversity_records
print(f"Total before dedup: {len(all_records)} "
      f"(core: {len(core_records)}, diversity: {len(diversity_records)})")

seen = set()
deduped_records = []
for r in all_records:
    h = hashlib.md5(normalize_for_dedup(r["solution"]).encode()).hexdigest()
    if h in seen:
        continue
    seen.add(h)
    deduped_records.append(r)

print(f"After dedup: {len(deduped_records)} "
      f"({len(all_records) - len(deduped_records)} duplicates removed)")

Total before dedup: 315000 (core: 220000, diversity: 95000)
After dedup: 280317 (34683 duplicates removed)


## 6. Decontaminate against HumanEval + MBPP

**Mandatory if you plan to report HumanEval/MBPP as evaluation metrics.** Pulls both benchmarks live and drops any SFT sample whose normalized problem+solution text overlaps with an eval item.

In [13]:
def load_eval_contamination_set():
    fingerprints = set()
    try:
        he = load_dataset("openai_humaneval", split="test")
        for row in he:
            key = normalize_for_dedup(row.get("prompt", "") + row.get("canonical_solution", ""))
            fingerprints.add(key[:300])
    except Exception as e:
        print(f"  WARNING: could not load HumanEval: {e}")

    try:
        mbpp = load_dataset("mbpp", split="test")
        for row in mbpp:
            key = normalize_for_dedup(row.get("text", "") + row.get("code", ""))
            fingerprints.add(key[:300])
    except Exception as e:
        print(f"  WARNING: could not load MBPP: {e}")

    print(f"Decontamination set built: {len(fingerprints)} fingerprints")
    return fingerprints


def is_contaminated(record, contamination_set):
    key = normalize_for_dedup(record["problem"] + record["solution"])[:300]
    return key in contamination_set

In [14]:
if not CONFIG["skip_decontamination"]:
    contamination_set = load_eval_contamination_set()
    clean_records = [r for r in deduped_records if not is_contaminated(r, contamination_set)]
    print(f"After decontamination: {len(clean_records)} "
          f"({len(deduped_records) - len(clean_records)} contaminated samples removed)")
else:
    clean_records = deduped_records
    print("Skipped decontamination (CONFIG['skip_decontamination'] = True)")

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.06k [00:00<?, ?B/s]

Decontamination set built: 0 fingerprints
After decontamination: 280317 (0 contaminated samples removed)


### 7. Inspect the final mix

In [15]:
layer_counts = Counter(r["layer"] for r in clean_records)
total = len(clean_records)
print(f"Final dataset: {total} examples\n")
for layer, count in layer_counts.items():
    print(f"  {layer:10s}: {count:>7,}  ({100*count/total:.1f}%)")

print()
source_counts = Counter(r["source"] for r in clean_records)
for source, count in source_counts.most_common():
    print(f"  {source:35s}: {count:>7,}")

total_tokens = sum(len(r["text"]) // 4 for r in clean_records)
print(f"\nApprox total tokens: {total_tokens:,}")

Final dataset: 280317 examples

  core      : 206,146  (73.5%)
  diversity :  74,171  (26.5%)

  opencodeinstruct                   :  89,211
  tested_143k                        :  66,960
  sc2_exec_filter                    :  49,975
  magicoder_evol_instruct            :  27,130
  opencoder_sft_package_instruct     :  24,992
  opencoder_sft_educational_instruct :  22,049

Approx total tokens: 99,071,755


Quick sanity check — look at a random example from each layer:

In [16]:
import random

for layer in ["core", "diversity"]:
    candidates = [r for r in clean_records if r["layer"] == layer]
    if candidates:
        sample = random.choice(candidates)
        print(f"=== {layer.upper()} sample (source: {sample['source']}) ===")
        print(sample["text"][:600])
        print("...\n")

=== CORE sample (source: tested_143k) ===
### Problem
You are given a Python code snippet representing a list of database comparisons. Each comparison is represented as an object with attributes `id`, `state`, `time`, and `comparison_type_id`. The `time` attribute is a `datetime` object representing the time of the comparison. Your task is to write a function that takes this list of comparisons as input and returns the comparison with the earliest time for each unique `comparison_type_id`.

Write a function `earliest_comparisons(comparisons)` where:
- comparisons: a list of database comparisons represented as objects with attributes `
...

=== DIVERSITY sample (source: opencoder_sft_package_instruct) ===
### Problem
**

In this problem, you are required to write a function named `preprocess_dataset_float32_binarize` for preprocessing a given dataset. The function should perform the following tasks:

1. Convert the features of the training and test sets to `float32` datatype.
2. Binariz

## 8. Save final training-ready JSONL

In [17]:
with open(CONFIG["out_path"], "w") as f:
    for i, r in enumerate(clean_records):
        r["id"] = i
        f.write(json.dumps(r) + "\n")

print(f"Wrote {len(clean_records)} examples to {CONFIG['out_path']}")

Wrote 280317 examples to sft_mix_final.jsonl


## Next steps

1. Verify the printed record counts per source look sane (0 records = a field name changed upstream — re-check `ds.column_names` for that loader).
2. Check the core/diversity ratio in the summary above — target is ~65/35, but dedup/decontamination may shift it. Adjust `CONFIG` sizes and re-run if it drifted too far.
3. Feed `sft_mix_final.jsonl`'s `text` field into your SFT training loop.
4. Keep this notebook's output cell counts as a record for your paper's data-provenance section (source breakdown + decontamination count).

## 9. Inspect the saved file directly

Open `sft_mix_final.jsonl` from disk and print what's actually stored inside — a plain read-back, independent of the `clean_records` variable still in memory, so this also works if you're resuming a fresh runtime that only has the saved file.

In [ ]:
import json

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

loaded = load_jsonl(CONFIG["out_path"])
print(f"Total records in {CONFIG['out_path']}: {len(loaded)}")
print(f"\nFields stored per record:")
for k, v in loaded[0].items():
    preview = str(v)
    if len(preview) > 80:
        preview = preview[:80] + "..."
    print(f"  {k:10s}: {type(v).__name__:8s}  example -> {preview}")

### 9a. Print the first N records in full

In [ ]:
N = 5

for r in loaded[:N]:
    print("-" * 70)
    print(f"id       : {r['id']}")
    print(f"source   : {r['source']}")
    print(f"layer    : {r['layer']}")
    print(f"problem  :\n  {r['problem'][:300]}")
    print(f"solution :\n  {r['solution'][:300]}")
    if r.get("tests"):
        print(f"tests    :\n  {str(r['tests'])[:300]}")
    print(f"text (raw training string):\n  {r['text'][:400]}")
    print()

### 9b. Print a few random records instead

In [ ]:
import random

random_sample = random.sample(loaded, min(5, len(loaded)))
for r in random_sample:
    print("-" * 70)
    print(f"id: {r['id']}  |  source: {r['source']}  |  layer: {r['layer']}")
    print(f"PROBLEM:\n{r['problem'][:300]}\n")
    print(f"SOLUTION:\n{r['solution'][:300]}\n")

### 9c. Look up one specific record by id, or filter by source/layer

In [ ]:
# --- look up a specific id ---
target_id = 0
match = next((r for r in loaded if r["id"] == target_id), None)
if match:
    print(json.dumps(match, indent=2)[:1500])
else:
    print(f"No record with id={target_id}")

In [ ]:
# --- filter by source and/or layer ---
filter_source = "magicoder_evol_instruct"   # set to None to skip this filter
filter_layer = None                          # "core" / "diversity" / None

filtered = loaded
if filter_source:
    filtered = [r for r in filtered if r["source"] == filter_source]
if filter_layer:
    filtered = [r for r in filtered if r["layer"] == filter_layer]

print(f"Matched {len(filtered)} records")
for r in filtered[:3]:
    print("-" * 70)
    print(f"id: {r['id']}  |  source: {r['source']}  |  layer: {r['layer']}")
    print(f"PROBLEM: {r['problem'][:200]}")
    print(f"SOLUTION: {r['solution'][:200]}")

### 9d. Full statistical summary of what's stored

In [ ]:
from collections import Counter

layer_counts = Counter(r["layer"] for r in loaded)
source_counts = Counter(r["source"] for r in loaded)
total = len(loaded)

print(f"=== {CONFIG['out_path']} contents summary ===\n")
print(f"Total records: {total}\n")

print("By layer:")
for layer, count in layer_counts.items():
    print(f"  {layer:10s}: {count:>7,} ({100*count/total:.1f}%)")

print("\nBy source:")
for source, count in source_counts.most_common():
    print(f"  {source:35s}: {count:>7,} ({100*count/total:.1f}%)")

lengths = [len(r["solution"]) for r in loaded]
print(f"\nSolution length (chars): min={min(lengths)}  "
      f"median={sorted(lengths)[len(lengths)//2]}  max={max(lengths)}")

total_tokens = sum(len(r["text"]) // 4 for r in loaded)
print(f"Approx total tokens: {total_tokens:,}")

has_tests = sum(1 for r in loaded if r.get("tests"))
print(f"Records with tests populated: {has_tests}/{total} ({100*has_tests/total:.1f}%)")